# Weather Classification — Watching Where the Model Looks During Training

Fine-tunes a CNN on the [Weather Image Recognition](https://www.kaggle.com/datasets/jehanbhathena/weather-dataset) dataset and records **Grad-CAM**, **Grad-CAM++** and **Score-CAM** heatmaps on a fixed set of images after every epoch, so the evolution of the model's attention can be inspected while it learns.

# 1. Setup

## 1.1 Imports

In [ ]:
import os
import gc
import copy
import math
import time
import random
import warnings
from io import BytesIO
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from PIL import Image, ImageFile
from tqdm.auto import tqdm
from IPython.display import display, Image as IPImage
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    top_k_accuracy_score,
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.flop_counter import FlopCounterMode
import torchvision
import torchvision.transforms as T

ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
print(f"torch {torch.__version__} | torchvision {torchvision.__version__} | GPUs: {torch.cuda.device_count()}")

## 1.2 Configuration

In [ ]:
class CFG:
    SEED = 42
    INPUT_ROOT = "/kaggle/input"
    DATA_DIR = None
    OUTPUT_DIR = "/kaggle/working"
    CHECKPOINT_NAME = "cam_weather_best.pth"
    IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp", ".gif")

    VAL_SIZE = 0.15
    TEST_SIZE = 0.15
    SAMPLES_PER_CLASS = 6

    IMG_SIZE = 224
    CACHE_IMAGES = True
    CACHE_SHORT_SIDE = 256
    IO_THREADS = 16
    NORM_STATS = "imagenet"
    IMAGENET_MEAN = (0.485, 0.456, 0.406)
    IMAGENET_STD = (0.229, 0.224, 0.225)
    RRC_SCALE = (0.5, 1.0)
    AUTO_AUGMENT = "trivial"
    RANDOM_ERASING = 0.1

    BACKBONE = "resnet50"
    PRETRAINED = True
    DROP_RATE = 0.3

    EPOCHS = 15
    FREEZE_EPOCHS = 2
    WARMUP_EPOCHS = 1
    BATCH_SIZE = 64
    EVAL_BATCH_SIZE = 128
    NUM_WORKERS = os.cpu_count()
    HEAD_LR = 1e-3
    BACKBONE_LR = 1e-4
    MIN_LR_RATIO = 0.01
    WEIGHT_DECAY = 1e-4
    LABEL_SMOOTHING = 0.1
    USE_AMP = True
    GRAD_CLIP = 2.0
    MONITOR = "val_f1"
    PATIENCE = 6
    MULTI_GPU = True

    CAM_METHODS = ("gradcam", "gradcam++", "scorecam")
    CAM_TARGET_DURING_TRAINING = "true"
    SNAPSHOT_PER_CLASS = 1
    SCORECAM_TOP_K = 512
    SCORECAM_BATCH = 128
    CAM_ALPHA = 0.45
    CAM_COLORMAP = "jet"
    EVOLUTION_MAX_COLS = 8
    GIF_FPS = 2
    N_GIFS_TO_SHOW = 4
    ERROR_CAM_METHOD = "gradcam++"
    N_ERROR_CAMS = 6

    TOP_K = 3
    URLS = [
        "https://upload.wikimedia.org/wikipedia/commons/f/f1/A_drop_of_dew_on_a_clover.jpg",
        "https://upload.wikimedia.org/wikipedia/commons/5/51/A_foggy_winter_morning.jpg",
        "https://upload.wikimedia.org/wikipedia/commons/b/b3/Saint-Amant_16_Gel%C3%A9e_blanche_2008.jpg",
        "https://upload.wikimedia.org/wikipedia/commons/b/ba/Ice_on_grass.jpg",
        "https://upload.wikimedia.org/wikipedia/commons/9/94/Granizo.jpg",
        "https://upload.wikimedia.org/wikipedia/commons/c/c2/Port_and_lighthouse_overnight_storm_with_lightning_in_Port-la-Nouvelle.jpg",
        "https://upload.wikimedia.org/wikipedia/commons/6/65/Hard_rain_on_a_roof.jpg",
        "https://upload.wikimedia.org/wikipedia/commons/5/5c/Double-alaskan-rainbow.jpg",
        "https://upload.wikimedia.org/wikipedia/commons/8/8b/Rime_ice.jpg",
        "https://upload.wikimedia.org/wikipedia/commons/2/2d/Sandsturm_in_der_Namib_%282017%29.jpg",
        "https://upload.wikimedia.org/wikipedia/commons/1/12/CargoNet_Di_12_Euro_4000_L%C3%B8nsdal_-_Bolna.jpg",
    ]
    URL_USER_AGENT = "Mozilla/5.0 (compatible; WeatherCAM/1.0; educational Kaggle notebook)"


pd.DataFrame(
    [(k, v) for k, v in vars(CFG).items() if not k.startswith("_") and k != "URLS"],
    columns=["parameter", "value"],
)

## 1.3 Reproducibility & Devices

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


seed_everything(CFG.SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_GPUS = torch.cuda.device_count()
USE_DP = CFG.MULTI_GPU and N_GPUS > 1
AMP_ENABLED = CFG.USE_AMP and DEVICE.type == "cuda"
torch.backends.cudnn.benchmark = True
os.makedirs(CFG.OUTPUT_DIR, exist_ok=True)
GIF_DIR = os.path.join(CFG.OUTPUT_DIR, "cam_gifs")
os.makedirs(GIF_DIR, exist_ok=True)

print(f"Device: {DEVICE} | DataParallel: {USE_DP} | AMP: {AMP_ENABLED} | CPU workers: {CFG.NUM_WORKERS}")
for i in range(N_GPUS):
    props = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i} {props.name} | {props.total_memory / 1024**3:.1f} GB")


def unwrap(model):
    return model.module if isinstance(model, nn.DataParallel) else model


def maybe_parallel(model):
    return nn.DataParallel(model) if USE_DP else model

# 2. Data Analysis

## 2.1 Dataset Index

In [ ]:
def find_data_dir(root, required=("rainbow", "lightning", "sandstorm")):
    for dirpath, dirnames, _ in os.walk(root):
        if set(required).issubset({d.lower() for d in dirnames}):
            return dirpath
    raise FileNotFoundError(f"Weather dataset not found under {root}. Add 'jehanbhathena/weather-dataset' as notebook input.")


CFG.DATA_DIR = CFG.DATA_DIR or find_data_dir(CFG.INPUT_ROOT)
CFG.CLASSES = sorted(d for d in os.listdir(CFG.DATA_DIR) if os.path.isdir(os.path.join(CFG.DATA_DIR, d)))
CFG.NUM_CLASSES = len(CFG.CLASSES)
CLASS_TO_IDX = {c: i for i, c in enumerate(CFG.CLASSES)}

records = [
    {"path": str(p), "label": c}
    for c in CFG.CLASSES
    for p in sorted(Path(CFG.DATA_DIR, c).iterdir())
    if p.suffix.lower() in CFG.IMG_EXTENSIONS
]
df = pd.DataFrame(records)
df["target"] = df["label"].map(CLASS_TO_IDX)

print(f"Data dir: {CFG.DATA_DIR}")
print(f"Images: {len(df)} | Classes ({CFG.NUM_CLASSES}): {CFG.CLASSES}")
df.head()

## 2.2 Class Distribution

In [ ]:
counts = df["label"].value_counts().reindex(CFG.CLASSES)
dist = pd.DataFrame({"count": counts, "percent": (100 * counts / counts.sum()).round(2)})
display(dist.sort_values("count", ascending=False))
print(f"Imbalance ratio (max/min): {counts.max() / counts.min():.2f}")

fig, axes = plt.subplots(1, 2, figsize=(18, 5), gridspec_kw={"width_ratios": [2, 1]})
order = counts.sort_values(ascending=False).index
sns.barplot(x=order, y=counts[order].values, ax=axes[0], palette="viridis")
for i, v in enumerate(counts[order].values):
    axes[0].text(i, v, str(v), ha="center", va="bottom", fontsize=9)
axes[0].set_title("Images per class")
axes[0].tick_params(axis="x", rotation=45)
axes[1].pie(counts[order].values, labels=order, autopct="%1.1f%%", startangle=90, colors=sns.color_palette("viridis", len(order)))
axes[1].set_title("Class share")
plt.tight_layout()
plt.show()

## 2.3 Image Properties

In [ ]:
def read_header(path):
    size_kb = os.path.getsize(path) / 1024
    try:
        with Image.open(path) as im:
            return {"width": im.width, "height": im.height, "mode": im.mode, "format": im.format, "size_kb": size_kb}
    except Exception:
        return {"width": np.nan, "height": np.nan, "mode": "unreadable", "format": "unreadable", "size_kb": size_kb}


with ThreadPoolExecutor(CFG.IO_THREADS) as ex:
    meta = list(tqdm(ex.map(read_header, df["path"]), total=len(df), desc="Reading headers"))
df = pd.concat([df, pd.DataFrame(meta)], axis=1)
df["aspect_ratio"] = df["width"] / df["height"]
df["megapixels"] = df["width"] * df["height"] / 1e6

display(df[["width", "height", "aspect_ratio", "megapixels", "size_kb"]].describe().T.round(2))
display(pd.crosstab(df["label"], df["mode"], margins=True))
display(pd.crosstab(df["label"], df["format"], margins=True))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 11))
sns.scatterplot(data=df, x="width", y="height", hue="label", s=12, alpha=0.6, ax=axes[0, 0], palette="tab20")
axes[0, 0].set(xscale="log", yscale="log", title="Width vs height (log)")
axes[0, 0].legend(fontsize=7, ncol=2)
sns.histplot(df["aspect_ratio"].clip(0, 3), bins=60, ax=axes[0, 1], color="teal")
axes[0, 1].set_title("Aspect ratio (w/h, clipped at 3)")
sns.boxplot(data=df, x="label", y="megapixels", ax=axes[1, 0], palette="viridis", order=CFG.CLASSES)
axes[1, 0].set(yscale="log", title="Resolution per class (megapixels, log)")
axes[1, 0].tick_params(axis="x", rotation=45)
sns.boxplot(data=df, x="label", y="size_kb", ax=axes[1, 1], palette="magma", order=CFG.CLASSES)
axes[1, 1].set(yscale="log", title="File size per class (KB, log)")
axes[1, 1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

small = (df[["width", "height"]].min(axis=1) < CFG.IMG_SIZE).sum()
print(f"Images with short side < {CFG.IMG_SIZE}px: {small} ({100 * small / len(df):.1f}%)")

## 2.4 Image Loading, Validation & Cache

In [ ]:
def load_image(path, short_side=None):
    with Image.open(path) as im:
        if short_side and im.format == "JPEG":
            im.draft("RGB", (short_side, short_side))
        im = im.convert("RGB")
    if short_side:
        w, h = im.size
        scale = short_side / min(w, h)
        if scale < 1:
            im = im.resize((max(1, round(w * scale)), max(1, round(h * scale))), Image.BICUBIC)
    return im


def image_stats(im):
    rgb = np.asarray(im.resize((64, 64)), dtype=np.float32) / 255.0
    hsv = np.asarray(im.resize((64, 64)).convert("HSV"), dtype=np.float32) / 255.0
    m, sq = rgb.mean((0, 1)), (rgb**2).mean((0, 1))
    return {
        "mean_r": m[0], "mean_g": m[1], "mean_b": m[2],
        "sq_r": sq[0], "sq_g": sq[1], "sq_b": sq[2],
        "saturation": hsv[..., 1].mean(), "brightness": hsv[..., 2].mean(),
    }


def safe_load(path):
    try:
        im = load_image(path, CFG.CACHE_SHORT_SIDE)
        return im, image_stats(im)
    except Exception:
        return None, None


with ThreadPoolExecutor(CFG.IO_THREADS) as ex:
    loaded = list(tqdm(ex.map(safe_load, df["path"]), total=len(df), desc="Loading & validating"))

valid = np.array([im is not None for im, _ in loaded])
print(f"Unreadable images removed: {(~valid).sum()}")
for p in df.loc[~valid, "path"]:
    print("  ", p)

IMAGE_CACHE = {p: im for p, (im, _) in zip(df["path"], loaded) if im is not None} if CFG.CACHE_IMAGES else {}
stats_df = pd.DataFrame([s for _, s in loaded if s is not None])
df = pd.concat([df[valid].reset_index(drop=True), stats_df], axis=1)
del loaded
gc.collect()
print(f"Usable images: {len(df)} | Cached: {len(IMAGE_CACHE)}")

## 2.5 Sample Images per Class

In [ ]:
def get_image(path):
    return IMAGE_CACHE[path] if path in IMAGE_CACHE else load_image(path, CFG.CACHE_SHORT_SIDE)


n = CFG.SAMPLES_PER_CLASS
fig, axes = plt.subplots(CFG.NUM_CLASSES, n, figsize=(2.6 * n, 2.4 * CFG.NUM_CLASSES))
for r, c in enumerate(CFG.CLASSES):
    paths = df[df["label"] == c].sample(n, random_state=CFG.SEED)["path"].tolist()
    for k, p in enumerate(paths):
        ax = axes[r, k]
        ax.imshow(get_image(p))
        ax.axis("off")
        if k == 0:
            ax.set_title(c, loc="left", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## 2.6 Color Statistics per Class

In [ ]:
color = df.groupby("label")[["mean_r", "mean_g", "mean_b", "saturation", "brightness"]].mean().reindex(CFG.CLASSES)

fig, axes = plt.subplots(1, 3, figsize=(20, 5.5), gridspec_kw={"width_ratios": [1.1, 1, 1]})
sns.heatmap(color, annot=True, fmt=".2f", cmap="coolwarm", ax=axes[0])
axes[0].set_title("Mean channel / HSV values per class")
sns.boxplot(data=df, x="label", y="saturation", ax=axes[1], palette="Set2", order=CFG.CLASSES)
axes[1].set_title("Saturation per class")
axes[1].tick_params(axis="x", rotation=45)
sns.boxplot(data=df, x="label", y="brightness", ax=axes[2], palette="Set3", order=CFG.CLASSES)
axes[2].set_title("Brightness per class")
axes[2].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(18, 1.6))
ax.imshow(color[["mean_r", "mean_g", "mean_b"]].values[None], aspect="auto")
ax.set_xticks(range(CFG.NUM_CLASSES), CFG.CLASSES)
ax.set_yticks([])
ax.set_title("Average color per class")
plt.show()

## 2.7 Train / Validation / Test Split

In [ ]:
train_df, temp_df = train_test_split(
    df, test_size=CFG.VAL_SIZE + CFG.TEST_SIZE, stratify=df["target"], random_state=CFG.SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=CFG.TEST_SIZE / (CFG.VAL_SIZE + CFG.TEST_SIZE), stratify=temp_df["target"], random_state=CFG.SEED
)
train_df, val_df, test_df = (d.reset_index(drop=True) for d in (train_df, val_df, test_df))

split_table = pd.DataFrame({
    "train": train_df["label"].value_counts(),
    "val": val_df["label"].value_counts(),
    "test": test_df["label"].value_counts(),
}).reindex(CFG.CLASSES)
split_table.loc["TOTAL"] = split_table.sum()
display(split_table)

## 2.8 Normalization Statistics

In [ ]:
if CFG.NORM_STATS == "dataset":
    mean = train_df[["mean_r", "mean_g", "mean_b"]].mean().values
    std = np.sqrt(train_df[["sq_r", "sq_g", "sq_b"]].mean().values - mean**2)
    MEAN, STD = tuple(float(v) for v in mean.round(4)), tuple(float(v) for v in std.round(4))
else:
    MEAN, STD = CFG.IMAGENET_MEAN, CFG.IMAGENET_STD
print(f"Normalization ({CFG.NORM_STATS}) -> mean: {MEAN} | std: {STD}")

## 2.9 Transforms, Datasets & DataLoaders

In [ ]:
BICUBIC = T.InterpolationMode.BICUBIC


def build_transforms(train):
    if train:
        ops = [T.RandomResizedCrop(CFG.IMG_SIZE, scale=CFG.RRC_SCALE, interpolation=BICUBIC), T.RandomHorizontalFlip()]
        if CFG.AUTO_AUGMENT == "trivial":
            ops.append(T.TrivialAugmentWide())
        elif CFG.AUTO_AUGMENT == "randaugment":
            ops.append(T.RandAugment(num_ops=2, magnitude=9))
        ops += [T.ToTensor(), T.Normalize(MEAN, STD)]
        if CFG.RANDOM_ERASING > 0:
            ops.append(T.RandomErasing(p=CFG.RANDOM_ERASING, value="random"))
        return T.Compose(ops)
    return T.Compose([
        T.Resize(int(CFG.IMG_SIZE / 0.875), interpolation=BICUBIC),
        T.CenterCrop(CFG.IMG_SIZE),
        T.ToTensor(),
        T.Normalize(MEAN, STD),
    ])


DISPLAY_TRANSFORM = T.Compose([T.Resize(int(CFG.IMG_SIZE / 0.875), interpolation=BICUBIC), T.CenterCrop(CFG.IMG_SIZE)])


def denormalize(t):
    t = t.detach().cpu() * torch.tensor(STD).view(3, 1, 1) + torch.tensor(MEAN).view(3, 1, 1)
    return t.clamp(0, 1).permute(1, 2, 0).numpy()


class WeatherDataset(Dataset):
    def __init__(self, frame, transform):
        self.paths = frame["path"].tolist()
        self.targets = frame["target"].tolist()
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        return self.transform(get_image(self.paths[idx])), self.targets[idx]


def make_loader(frame, train, batch_size=None):
    return DataLoader(
        WeatherDataset(frame, build_transforms(train)),
        batch_size=batch_size or (CFG.BATCH_SIZE if train else CFG.EVAL_BATCH_SIZE),
        shuffle=train,
        num_workers=CFG.NUM_WORKERS,
        pin_memory=DEVICE.type == "cuda",
        drop_last=train,
        persistent_workers=CFG.NUM_WORKERS > 0,
    )


train_loader = make_loader(train_df, train=True)
val_loader = make_loader(val_df, train=False)
test_loader = make_loader(test_df, train=False)
xb, yb = next(iter(train_loader))
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")
print(f"Batch: {tuple(xb.shape)} | dtype: {xb.dtype} | range: [{xb.min():.2f}, {xb.max():.2f}]")

## 2.10 Augmentation Preview

In [ ]:
aug = build_transforms(train=True)
preview = train_df.groupby("label").sample(1, random_state=CFG.SEED).head(4)
fig, axes = plt.subplots(len(preview), 7, figsize=(19, 2.9 * len(preview)))
for r, (_, row) in enumerate(preview.iterrows()):
    img = get_image(row["path"])
    axes[r, 0].imshow(DISPLAY_TRANSFORM(img))
    axes[r, 0].set_title(f"original: {row['label']}", fontsize=10)
    for k in range(1, 7):
        axes[r, k].imshow(denormalize(aug(img)))
        axes[r, k].set_title(f"augmented #{k}", fontsize=10)
    for ax in axes[r]:
        ax.axis("off")
plt.tight_layout()
plt.show()

# 3. Model & Explainability Tools

## 3.1 Backbone Factory

Any backbone in `BACKBONES` can be selected through `CFG.BACKBONE`; `target_layer` is the last convolutional feature map used by the CAM methods.

In [ ]:
BACKBONES = {
    "resnet50": {"target_layer": "layer4", "head": "fc"},
    "efficientnet_b0": {"target_layer": "features.8", "head": "classifier"},
    "convnext_tiny": {"target_layer": "features.7", "head": "classifier"},
}


def build_model(name=None, num_classes=None, pretrained=None):
    name = name or CFG.BACKBONE
    num_classes = num_classes or CFG.NUM_CLASSES
    pretrained = CFG.PRETRAINED if pretrained is None else pretrained
    try:
        model = torchvision.models.get_model(name, weights="DEFAULT" if pretrained else None)
    except Exception as e:
        print(f"Pretrained weights unavailable ({e.__class__.__name__}); enable Internet in the notebook. Using random init.")
        model = torchvision.models.get_model(name, weights=None)
    if name.startswith("resnet"):
        model.fc = nn.Sequential(nn.Dropout(CFG.DROP_RATE), nn.Linear(model.fc.in_features, num_classes))
    else:
        last = model.classifier[-1]
        model.classifier[-1] = nn.Sequential(nn.Dropout(CFG.DROP_RATE), nn.Linear(last.in_features, num_classes))
    return model


def get_target_layer(model):
    return unwrap(model).get_submodule(BACKBONES[CFG.BACKBONE]["target_layer"])


def get_head(model):
    return unwrap(model).get_submodule(BACKBONES[CFG.BACKBONE]["head"])


def set_backbone_trainable(model, trainable):
    net = unwrap(model)
    for p in net.parameters():
        p.requires_grad_(trainable)
    for p in get_head(net).parameters():
        p.requires_grad_(True)

## 3.2 CAM Methods: Grad-CAM, Grad-CAM++, Score-CAM

- **Grad-CAM**: channel weights = spatial mean of the gradients of the class score w.r.t. the target feature map.
- **Grad-CAM++**: pixel-wise weighting of positive gradients (higher-order terms), better for multiple instances of the same object.
- **Score-CAM**: gradient-free; each upsampled activation map masks the input and its weight is the resulting class probability.

In [ ]:
class CAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None
        self.handle = None

    def __enter__(self):
        self.handle = self.target_layer.register_forward_hook(self._forward_hook)
        return self

    def __exit__(self, *args):
        self.handle.remove()
        self.activations, self.gradients = None, None

    def _forward_hook(self, module, inputs, output):
        self.activations = output
        if output.requires_grad:
            output.register_hook(self._save_gradient)

    def _save_gradient(self, grad):
        self.gradients = grad

    @staticmethod
    def normalize(cam, size):
        cam = F.interpolate(cam.unsqueeze(1).float(), size=size, mode="bilinear", align_corners=False).squeeze(1)
        flat = cam.flatten(1)
        mn, mx = flat.min(1)[0].view(-1, 1, 1), flat.max(1)[0].view(-1, 1, 1)
        return (cam - mn) / (mx - mn + 1e-8)

    def gradient_cams(self, x, target=None, methods=("gradcam", "gradcam++")):
        self.model.zero_grad(set_to_none=True)
        with torch.enable_grad():
            x = x.clone().requires_grad_(True)
            logits = self.model(x)
            idx = logits.argmax(1) if target is None else target
            logits.gather(1, idx.view(-1, 1)).sum().backward()
        A, G = self.activations.detach().float(), self.gradients.detach().float()
        maps = {}
        if "gradcam" in methods:
            weights = G.mean((2, 3), keepdim=True)
            maps["gradcam"] = self.normalize(F.relu((weights * A).sum(1)), x.shape[-2:])
        if "gradcam++" in methods:
            g2, g3 = G.pow(2), G.pow(3)
            denom = 2 * g2 + A.sum((2, 3), keepdim=True) * g3
            alpha = g2 / torch.where(denom != 0, denom, torch.ones_like(denom))
            weights = (alpha * F.relu(G)).sum((2, 3), keepdim=True)
            maps["gradcam++"] = self.normalize(F.relu((weights * A).sum(1)), x.shape[-2:])
        self.model.zero_grad(set_to_none=True)
        return maps, logits.detach().float().softmax(1), idx

    @torch.no_grad()
    def scorecam(self, x, target=None, top_k=None, batch_size=128):
        logits = self.model(x)
        A = self.activations.float()
        idx = logits.argmax(1) if target is None else target
        cams = []
        for i in range(x.size(0)):
            acts = A[i]
            if top_k and top_k < acts.size(0):
                acts = acts[acts.mean((1, 2)).topk(top_k).indices]
            masks = F.interpolate(acts.unsqueeze(1), size=x.shape[-2:], mode="bilinear", align_corners=False).squeeze(1)
            flat = masks.flatten(1)
            mn, mx = flat.min(1)[0].view(-1, 1, 1), flat.max(1)[0].view(-1, 1, 1)
            masks = (masks - mn) / (mx - mn).clamp_min(1e-8)
            scores = []
            for s in range(0, masks.size(0), batch_size):
                with torch.autocast(device_type=x.device.type, dtype=torch.float16, enabled=AMP_ENABLED):
                    out = self.model(x[i:i + 1] * masks[s:s + batch_size].unsqueeze(1))
                scores.append(out.float().softmax(1)[:, idx[i]])
            scores = torch.cat(scores)
            cams.append(F.relu((scores.view(-1, 1, 1) * acts).sum(0)))
        return self.normalize(torch.stack(cams), x.shape[-2:]), logits.float().softmax(1), idx


def compute_cams(model, x, target=None, methods=None):
    methods = methods or CFG.CAM_METHODS
    net = unwrap(model)
    was_training = net.training
    net.eval()
    x = x.to(DEVICE)
    target = target.to(DEVICE) if target is not None else None
    maps, probs, idx = {}, None, None
    with CAM(net, get_target_layer(net)) as cam:
        grad_methods = [m for m in methods if m != "scorecam"]
        if grad_methods:
            out, probs, idx = cam.gradient_cams(x, target, grad_methods)
            maps.update(out)
        if "scorecam" in methods:
            out, probs, idx = cam.scorecam(x, target, CFG.SCORECAM_TOP_K, CFG.SCORECAM_BATCH)
            maps["scorecam"] = out
    net.train(was_training)
    return {k: v.cpu().numpy() for k, v in maps.items()}, probs.cpu().numpy(), idx.cpu().numpy()

## 3.3 Visualization Helpers

In [ ]:
METHOD_TITLES = {"gradcam": "Grad-CAM", "gradcam++": "Grad-CAM++", "scorecam": "Score-CAM"}


def overlay(img, cam, alpha=None):
    alpha = CFG.CAM_ALPHA if alpha is None else alpha
    heat = plt.get_cmap(CFG.CAM_COLORMAP)(cam)[..., :3]
    return np.clip((1 - alpha) * img + alpha * heat, 0, 1)


def cam_entropy(cam):
    p = cam.reshape(len(cam), -1).astype(np.float64) + 1e-12
    p = p / p.sum(1, keepdims=True)
    return -(p * np.log(p)).sum(1) / np.log(p.shape[1])


def fig_to_pil(fig):
    fig.canvas.draw()
    return Image.fromarray(np.asarray(fig.canvas.buffer_rgba())[..., :3].copy())


def plot_cam_row(axes, img, maps, methods, titles=None):
    axes[0].imshow(img)
    axes[0].set_title(titles[0] if titles else "image", fontsize=10)
    for k, m in enumerate(methods, start=1):
        axes[k].imshow(overlay(img, maps[m]))
        axes[k].set_title(titles[k] if titles else METHOD_TITLES[m], fontsize=10)
    for ax in axes:
        ax.axis("off")

# 4. Model Info

## 4.1 Parameters & FLOPs

In [ ]:
def count_params(module, trainable_only=False):
    return sum(p.numel() for p in module.parameters() if p.requires_grad or not trainable_only)


@torch.no_grad()
def count_flops(model, img_size):
    model.eval()
    with FlopCounterMode(display=False) as counter:
        model(torch.zeros(1, 3, img_size, img_size, device=next(model.parameters()).device))
    return counter.get_total_flops()


model = build_model()
set_backbone_trainable(model, False)
trainable_frozen = count_params(model, True)
set_backbone_trainable(model, True)
display(pd.DataFrame({
    "backbone": CFG.BACKBONE,
    "pretrained": CFG.PRETRAINED,
    "params (M)": count_params(model) / 1e6,
    "trainable, frozen phase (M)": trainable_frozen / 1e6,
    "trainable, full fine-tune (M)": count_params(model, True) / 1e6,
    "GFLOPs": count_flops(model, CFG.IMG_SIZE) / 1e9,
}, index=["model"]).round(3))

## 4.2 Top-level Modules

In [ ]:
@torch.no_grad()
def top_level_summary(model, img_size):
    rows, hooks = [], []
    for name, module in model.named_children():
        hooks.append(module.register_forward_hook(
            lambda m, inp, out, name=name: rows.append({
                "module": name,
                "type": type(m).__name__,
                "output_shape": tuple(out.shape),
                "params (M)": round(count_params(m) / 1e6, 3),
            })
        ))
    model.eval()(torch.zeros(1, 3, img_size, img_size))
    for h in hooks:
        h.remove()
    return pd.DataFrame(rows)


display(top_level_summary(model, CFG.IMG_SIZE))
print(get_head(model))

## 4.3 Target Layer & CAM Cost

In [ ]:
model = model.to(DEVICE)
probe = torch.stack([build_transforms(False)(get_image(p)) for p in val_df["path"].head(2)])
with CAM(model, get_target_layer(model)) as cam, torch.no_grad():
    model.eval()(probe.to(DEVICE))
    feat_shape = tuple(cam.activations.shape)
print(f"Target layer: {BACKBONES[CFG.BACKBONE]['target_layer']} -> {feat_shape} "
      f"(channels={feat_shape[1]}, grid={feat_shape[2]}x{feat_shape[3]})")

timing = []
for m in CFG.CAM_METHODS:
    t0 = time.perf_counter()
    maps, _, _ = compute_cams(model, probe, methods=(m,))
    timing.append({"method": METHOD_TITLES[m], "ms / image": 1000 * (time.perf_counter() - t0) / len(probe),
                   "map shape": maps[m].shape[1:]})
display(pd.DataFrame(timing).round(1))
del model
gc.collect()
torch.cuda.empty_cache()

# 5. Training (with Attention Snapshots)

## 5.1 Snapshot Images

A fixed set of validation images (one per class by default) is explained after every epoch. Epoch 0 is the model before any fine-tuning (pretrained backbone + random head).

In [ ]:
snap_df = val_df.groupby("label", sort=False).sample(CFG.SNAPSHOT_PER_CLASS, random_state=CFG.SEED)
snap_df = snap_df.sort_values("target").reset_index(drop=True)
X_SNAP = torch.stack([build_transforms(False)(get_image(p)) for p in snap_df["path"]])
Y_SNAP = torch.tensor(snap_df["target"].values)
SNAP_IMAGES = [denormalize(x) for x in X_SNAP]
SNAPSHOTS = []


def take_snapshot(model, epoch):
    target = Y_SNAP if CFG.CAM_TARGET_DURING_TRAINING == "true" else None
    maps, probs, _ = compute_cams(model, X_SNAP, target)
    SNAPSHOTS.append({
        "epoch": epoch,
        "maps": {k: v.astype(np.float16) for k, v in maps.items()},
        "probs": probs,
    })


fig, axes = plt.subplots(1, len(snap_df), figsize=(2.2 * len(snap_df), 2.6))
for ax, img, label in zip(np.atleast_1d(axes), SNAP_IMAGES, snap_df["label"]):
    ax.imshow(img)
    ax.set_title(label, fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 5.2 Loss, Optimizer & Scheduler

In [ ]:
def build_optimizer(model):
    head_ids = {id(p) for p in get_head(model).parameters()}
    head = [p for p in model.parameters() if id(p) in head_ids]
    backbone = [p for p in model.parameters() if id(p) not in head_ids]
    return torch.optim.AdamW([
        {"params": backbone, "lr": CFG.BACKBONE_LR, "name": "backbone"},
        {"params": head, "lr": CFG.HEAD_LR, "name": "head"},
    ], weight_decay=CFG.WEIGHT_DECAY)


def warmup_cosine(optimizer, total_steps, warmup_steps, min_ratio):
    def fn(step):
        if step < warmup_steps:
            return (step + 1) / max(1, warmup_steps)
        progress = min(1.0, (step - warmup_steps) / max(1, total_steps - warmup_steps))
        return min_ratio + (1 - min_ratio) * 0.5 * (1 + math.cos(math.pi * progress))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, fn)

## 5.3 Train & Validation Loops

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scheduler, scaler):
    model.train()
    loss_sum, correct, seen = 0.0, 0, 0
    pbar = tqdm(loader, desc="train", leave=False)
    for x, y in pbar:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP_ENABLED):
            logits = model(x)
        loss = criterion(logits.float(), y)

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], CFG.GRAD_CLIP)
        scale_before = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        if scaler.get_scale() >= scale_before:
            scheduler.step()

        bs = y.size(0)
        loss_sum += loss.item() * bs
        correct += (logits.argmax(1) == y).sum().item()
        seen += bs
        pbar.set_postfix(loss=f"{loss_sum / seen:.4f}")
    return loss_sum / seen, correct / seen


@torch.no_grad()
def predict(model, loader, desc="predict"):
    model.eval()
    all_probs, all_targets = [], []
    for x, y in tqdm(loader, desc=desc, leave=False):
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP_ENABLED):
            all_probs.append(model(x.to(DEVICE, non_blocking=True)).float().softmax(1).cpu())
        all_targets.append(y)
    return torch.cat(all_probs).numpy(), torch.cat(all_targets).numpy()


def compute_metrics(probs, targets):
    preds = probs.argmax(1)
    nll = -np.log(np.clip(probs[np.arange(len(targets)), targets], 1e-12, 1.0)).mean()
    return {"loss": float(nll), "acc": accuracy_score(targets, preds), "f1": f1_score(targets, preds, average="macro")}

## 5.4 Run Training

In [ ]:
def fit():
    seed_everything(CFG.SEED)
    SNAPSHOTS.clear()
    net = build_model().to(DEVICE)
    model = maybe_parallel(net)
    criterion = nn.CrossEntropyLoss(label_smoothing=CFG.LABEL_SMOOTHING)
    optimizer = build_optimizer(net)
    steps = len(train_loader)
    scheduler = warmup_cosine(optimizer, CFG.EPOCHS * steps, CFG.WARMUP_EPOCHS * steps, CFG.MIN_LR_RATIO)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)
    ckpt_path = os.path.join(CFG.OUTPUT_DIR, CFG.CHECKPOINT_NAME)

    take_snapshot(net, 0)
    history, best_score, best_epoch, bad_epochs = [], -np.inf, 0, 0
    print(f"{'epoch':>5} | {'phase':>8} | {'train_loss':>10} | {'train_acc':>9} | {'val_loss':>8} | {'val_acc':>7} | {'val_f1':>6} | {'time':>6}")
    for epoch in range(1, CFG.EPOCHS + 1):
        t0 = time.time()
        frozen = epoch <= CFG.FREEZE_EPOCHS
        set_backbone_trainable(net, not frozen)
        lrs = {g["name"]: g["lr"] for g in optimizer.param_groups}
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, scheduler, scaler)
        val = compute_metrics(*predict(model, val_loader, desc="val"))
        take_snapshot(net, epoch)
        row = {"epoch": epoch, "phase": "head" if frozen else "full", "lr_head": lrs["head"], "lr_backbone": lrs["backbone"],
               "train_loss": train_loss, "train_acc": train_acc, "val_loss": val["loss"], "val_acc": val["acc"],
               "val_f1": val["f1"], "time": time.time() - t0}
        history.append(row)

        improved = row[CFG.MONITOR] > best_score
        if improved:
            best_score, best_epoch, bad_epochs = row[CFG.MONITOR], epoch, 0
            torch.save({"state_dict": net.state_dict(), "backbone": CFG.BACKBONE, "classes": CFG.CLASSES,
                        "epoch": epoch, "metrics": row}, ckpt_path)
        else:
            bad_epochs += 1
        print(f"{epoch:>5} | {row['phase']:>8} | {train_loss:>10.4f} | {train_acc:>9.4f} | {row['val_loss']:>8.4f} | "
              f"{row['val_acc']:>7.4f} | {row['val_f1']:>6.4f} | {row['time']:>5.1f}s {'*' if improved else ''}")
        if bad_epochs >= CFG.PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

    print(f"Best {CFG.MONITOR}: {best_score:.4f} at epoch {best_epoch} -> {ckpt_path}")
    return pd.DataFrame(history), ckpt_path


history, CKPT_PATH = fit()
history.to_csv(os.path.join(CFG.OUTPUT_DIR, "history.csv"), index=False)
np.savez_compressed(
    os.path.join(CFG.OUTPUT_DIR, "cam_snapshots.npz"),
    **{f"{s['epoch']}_{m}": v for s in SNAPSHOTS for m, v in s["maps"].items()},
    **{f"{s['epoch']}_probs": s["probs"] for s in SNAPSHOTS},
)
torch.cuda.empty_cache()

## 5.5 Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
axes[0].plot(history["epoch"], history["train_loss"], label="train")
axes[0].plot(history["epoch"], history["val_loss"], label="val")
axes[0].set_title("Loss")
axes[1].plot(history["epoch"], history["train_acc"], label="train acc")
axes[1].plot(history["epoch"], history["val_acc"], label="val acc")
axes[1].plot(history["epoch"], history["val_f1"], label="val macro-F1")
axes[1].set_title("Accuracy / F1")
axes[2].plot(history["epoch"], history["lr_head"], label="head")
axes[2].plot(history["epoch"], history["lr_backbone"], label="backbone")
axes[2].set(yscale="log", title="Learning rate (epoch start)")
for ax in axes:
    if CFG.FREEZE_EPOCHS:
        ax.axvspan(0.5, CFG.FREEZE_EPOCHS + 0.5, color="gray", alpha=0.12, label="frozen backbone")
    ax.set_xlabel("epoch")
    ax.legend()
plt.tight_layout()
plt.show()
display(history.round(4))

## 5.6 Attention Evolution Grids

Rows are the snapshot images, columns are epochs. The title shows the probability assigned to the true class (green when the prediction is correct).

In [ ]:
def pick_epochs(n_snapshots, max_cols):
    if n_snapshots <= max_cols:
        return list(range(n_snapshots))
    return sorted(set(np.linspace(0, n_snapshots - 1, max_cols).round().astype(int).tolist()))


def plot_evolution(method):
    cols = pick_epochs(len(SNAPSHOTS), CFG.EVOLUTION_MAX_COLS)
    n_rows = len(SNAP_IMAGES)
    fig, axes = plt.subplots(n_rows, len(cols) + 1, figsize=(2.1 * (len(cols) + 1), 2.2 * n_rows))
    axes = np.atleast_2d(axes)
    for r in range(n_rows):
        axes[r, 0].imshow(SNAP_IMAGES[r])
        axes[r, 0].set_title(snap_df["label"][r], fontsize=10, fontweight="bold")
        for c, s_idx in enumerate(cols, start=1):
            snap = SNAPSHOTS[s_idx]
            probs = snap["probs"][r]
            correct = probs.argmax() == Y_SNAP[r].item()
            axes[r, c].imshow(overlay(SNAP_IMAGES[r], snap["maps"][method][r].astype(np.float32)))
            axes[r, c].set_title(f"ep {snap['epoch']} | p={probs[Y_SNAP[r]]:.2f}", fontsize=9, color="green" if correct else "red")
    for ax in axes.ravel():
        ax.axis("off")
    fig.suptitle(f"{METHOD_TITLES[method]} evolution during training", fontsize=15, fontweight="bold")
    plt.tight_layout(rect=(0, 0, 1, 0.97))
    plt.show()


for method in CFG.CAM_METHODS:
    plot_evolution(method)

## 5.7 Attention Evolution GIFs

In [ ]:
def make_gif(i):
    methods = list(CFG.CAM_METHODS)
    frames = []
    for snap in SNAPSHOTS:
        fig, axes = plt.subplots(1, len(methods) + 1, figsize=(3.2 * (len(methods) + 1), 3.6))
        probs = snap["probs"][i]
        plot_cam_row(axes, SNAP_IMAGES[i], {m: snap["maps"][m][i].astype(np.float32) for m in methods}, methods)
        pred = CFG.CLASSES[probs.argmax()]
        fig.suptitle(f"epoch {snap['epoch']} | true: {snap_df['label'][i]} | pred: {pred} ({probs.max():.0%})",
                     color="green" if pred == snap_df["label"][i] else "red", fontsize=12)
        plt.tight_layout(rect=(0, 0, 1, 0.97))
        frames.append(fig_to_pil(fig))
        plt.close(fig)
    path = os.path.join(GIF_DIR, f"{i:02d}_{snap_df['label'][i]}.gif")
    durations = [int(1000 / CFG.GIF_FPS)] * (len(frames) - 1) + [2500]
    frames[0].save(path, save_all=True, append_images=frames[1:], duration=durations, loop=0)
    return path


gif_paths = [make_gif(i) for i in tqdm(range(len(SNAP_IMAGES)), desc="GIFs")]
print(f"Saved {len(gif_paths)} GIFs to {GIF_DIR}")
for p in gif_paths[:CFG.N_GIFS_TO_SHOW]:
    print(os.path.basename(p))
    display(IPImage(filename=p))

## 5.8 Attention Dynamics: Stability, Focus & Confidence

- **Stability**: Pearson correlation between each epoch's map and the final map (1 = attention already converged).
- **Focus**: normalized entropy of the map (lower = more concentrated attention).
- **Confidence**: probability of the true class per snapshot image.

In [ ]:
epochs = [s["epoch"] for s in SNAPSHOTS]
fig, axes = plt.subplots(1, 3, figsize=(21, 5.5), gridspec_kw={"width_ratios": [1, 1, 1.3]})
for m in CFG.CAM_METHODS:
    final = SNAPSHOTS[-1]["maps"][m].astype(np.float32).reshape(len(SNAP_IMAGES), -1)
    corr, ent = [], []
    for s in SNAPSHOTS:
        cur = s["maps"][m].astype(np.float32).reshape(len(SNAP_IMAGES), -1)
        corr.append(np.nanmean([np.corrcoef(a, b)[0, 1] for a, b in zip(cur, final)]))
        ent.append(cam_entropy(s["maps"][m].astype(np.float32)).mean())
    axes[0].plot(epochs, corr, marker="o", label=METHOD_TITLES[m])
    axes[1].plot(epochs, ent, marker="o", label=METHOD_TITLES[m])
axes[0].set(title="Stability: corr(map_epoch, map_final)", xlabel="epoch", ylim=(-0.1, 1.05), xticks=epochs)
axes[1].set(title="Focus: normalized entropy (lower = sharper)", xlabel="epoch", xticks=epochs)
axes[0].legend()
axes[1].legend()
conf = np.stack([s["probs"][np.arange(len(Y_SNAP)), Y_SNAP.numpy()] for s in SNAPSHOTS], axis=1)
sns.heatmap(conf, annot=len(epochs) <= 20, fmt=".2f", cmap="RdYlGn", vmin=0, vmax=1,
            xticklabels=epochs, yticklabels=snap_df["label"], ax=axes[2])
axes[2].set(title="p(true class) per snapshot image", xlabel="epoch")
plt.tight_layout()
plt.show()

# 6. Evaluation (Validation Set)

## 6.1 Load Best Model

In [ ]:
ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
best_model = build_model(ckpt["backbone"], len(ckpt["classes"]), pretrained=False)
best_model.load_state_dict(ckpt["state_dict"])
best_model = best_model.to(DEVICE).eval()
INFER = maybe_parallel(best_model)
print(f"Loaded epoch {ckpt['epoch']} | val_f1={ckpt['metrics']['val_f1']:.4f} | val_acc={ckpt['metrics']['val_acc']:.4f}")

## 6.2 Metrics & Classification Report

In [ ]:
def report(probs, targets, title):
    preds = probs.argmax(1)
    summary = {
        "accuracy": accuracy_score(targets, preds),
        f"top-{CFG.TOP_K} accuracy": top_k_accuracy_score(targets, probs, k=CFG.TOP_K, labels=range(CFG.NUM_CLASSES)),
        "macro F1": f1_score(targets, preds, average="macro"),
        "weighted F1": f1_score(targets, preds, average="weighted"),
        "NLL": compute_metrics(probs, targets)["loss"],
    }
    print(f"===== {title} =====")
    display(pd.DataFrame(summary, index=[title]).round(4))
    rep = classification_report(targets, preds, target_names=CFG.CLASSES, output_dict=True, digits=4)
    display(pd.DataFrame(rep).T.round(4).style.background_gradient(cmap="Greens", subset=["precision", "recall", "f1-score"]))
    return preds


val_probs, val_targets = predict(INFER, val_loader, desc="val")
val_preds = report(val_probs, val_targets, "Validation")

## 6.3 Confusion Matrix

In [ ]:
def plot_confusion(targets, preds, title):
    cm = confusion_matrix(targets, preds, labels=range(CFG.NUM_CLASSES))
    cm_norm = cm / cm.sum(1, keepdims=True).clip(min=1)
    fig, axes = plt.subplots(1, 2, figsize=(22, 8.5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CFG.CLASSES, yticklabels=CFG.CLASSES, ax=axes[0], cbar=False)
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", xticklabels=CFG.CLASSES, yticklabels=CFG.CLASSES, ax=axes[1], vmin=0, vmax=1)
    for ax, sub in zip(axes, ["counts", "row-normalized (recall)"]):
        ax.set(xlabel="predicted", ylabel="true", title=f"{title} — {sub}")
        ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    plt.show()


plot_confusion(val_targets, val_preds, "Validation")

## 6.4 Where Does the Model Look When It Fails?

For the most confident validation errors: the map for the **predicted** class next to the map for the **true** class.

In [ ]:
val_df["pred"] = val_preds
val_df["confidence"] = val_probs.max(1)
errors = val_df[val_df["pred"] != val_df["target"]].sort_values("confidence", ascending=False).head(CFG.N_ERROR_CAMS)
print(f"Validation errors: {(val_df['pred'] != val_df['target']).sum()} / {len(val_df)}")

if len(errors):
    m = CFG.ERROR_CAM_METHOD
    x = torch.stack([build_transforms(False)(get_image(p)) for p in errors["path"]])
    maps_pred, _, _ = compute_cams(best_model, x, torch.tensor(errors["pred"].values), methods=(m,))
    maps_true, _, _ = compute_cams(best_model, x, torch.tensor(errors["target"].values), methods=(m,))
    fig, axes = plt.subplots(len(errors), 3, figsize=(10, 3.3 * len(errors)))
    axes = np.atleast_2d(axes)
    for r, (_, row) in enumerate(errors.reset_index(drop=True).iterrows()):
        img = denormalize(x[r])
        axes[r, 0].imshow(img)
        axes[r, 0].set_title(f"true: {row['label']}", fontsize=10)
        axes[r, 1].imshow(overlay(img, maps_pred[m][r]))
        axes[r, 1].set_title(f"{METHOD_TITLES[m]} -> pred {CFG.CLASSES[row['pred']]} ({row['confidence']:.0%})", fontsize=10, color="red")
        axes[r, 2].imshow(overlay(img, maps_true[m][r]))
        axes[r, 2].set_title(f"{METHOD_TITLES[m]} -> true {row['label']}", fontsize=10, color="green")
        for ax in axes[r]:
            ax.axis("off")
    plt.tight_layout()
    plt.show()

# 7. Test (Hold-out Test Set)

## 7.1 Test Metrics

In [ ]:
test_probs, test_targets = predict(INFER, test_loader, desc="test")
test_preds = report(test_probs, test_targets, "Test")
test_df["pred"] = [CFG.CLASSES[i] for i in test_preds]
test_df["confidence"] = test_probs.max(1)
test_df.to_csv(os.path.join(CFG.OUTPUT_DIR, "test_predictions.csv"), index=False)

## 7.2 Test Confusion Matrix

In [ ]:
plot_confusion(test_targets, test_preds, "Test")

## 7.3 CAM Method Comparison (one image per class)

Maps are computed for the **predicted** class, as in deployment.

In [ ]:
cmp_df = test_df.groupby("label", sort=False).sample(1, random_state=CFG.SEED).sort_values("target").reset_index(drop=True)
x = torch.stack([build_transforms(False)(get_image(p)) for p in cmp_df["path"]])
maps, probs, idx = compute_cams(best_model, x)
methods = list(CFG.CAM_METHODS)
fig, axes = plt.subplots(len(cmp_df), len(methods) + 1, figsize=(3.1 * (len(methods) + 1), 3.2 * len(cmp_df)))
axes = np.atleast_2d(axes)
for r, row in cmp_df.iterrows():
    pred = CFG.CLASSES[idx[r]]
    titles = [f"T: {row['label']} | P: {pred} ({probs[r].max():.0%})"] + [METHOD_TITLES[m] for m in methods]
    plot_cam_row(axes[r], denormalize(x[r]), {m: maps[m][r] for m in methods}, methods, titles)
    axes[r, 0].title.set_color("green" if pred == row["label"] else "red")
plt.tight_layout()
plt.show()

# 8. Test with URL Images

## 8.1 URL Helpers

In [ ]:
def load_image_from_url(url, timeout=30):
    response = requests.get(url, headers={"User-Agent": CFG.URL_USER_AGENT}, timeout=timeout)
    response.raise_for_status()
    img = Image.open(BytesIO(response.content))
    img.draft("RGB", (CFG.CACHE_SHORT_SIDE, CFG.CACHE_SHORT_SIDE))
    return img.convert("RGB")


def explain_image(model, img, url=""):
    x = build_transforms(False)(img).unsqueeze(0)
    maps, probs, idx = compute_cams(model, x)
    probs = probs[0]
    top = np.argsort(probs)[::-1][:CFG.TOP_K]
    methods = list(CFG.CAM_METHODS)
    fig, axes = plt.subplots(1, len(methods) + 2, figsize=(3.6 * (len(methods) + 2), 3.8),
                             gridspec_kw={"width_ratios": [1] * (len(methods) + 1) + [1.1]})
    plot_cam_row(axes[:-1], denormalize(x[0]), {m: maps[m][0] for m in methods}, methods,
                 [f"P: {CFG.CLASSES[idx[0]]} ({probs.max():.0%})"] + [METHOD_TITLES[m] for m in methods])
    axes[-1].barh([CFG.CLASSES[i] for i in top][::-1], probs[top][::-1], color=sns.color_palette("viridis", CFG.TOP_K)[::-1])
    for i, v in enumerate(probs[top][::-1]):
        axes[-1].text(v + 0.01, i, f"{v:.1%}", va="center")
    axes[-1].set(xlim=(0, 1.2), title=f"Top-{CFG.TOP_K}")
    fig.suptitle(url.split("/")[-1][:90], fontsize=9, color="gray")
    plt.tight_layout(rect=(0, 0, 1, 0.97))
    plt.show()
    return CFG.CLASSES[idx[0]], float(probs.max())

## 8.2 Predictions & Explanations

Edit `CFG.URLS` (or the list below) with any image URLs.

In [ ]:
URLS = CFG.URLS

url_rows = []
for url in URLS:
    try:
        img = load_image_from_url(url)
    except Exception as e:
        print(f"Could not load {url}\n  -> {e}")
        continue
    pred, conf = explain_image(best_model, img, url)
    url_rows.append({"url": url.split("/")[-1][:60], "prediction": pred, "confidence": conf})

if url_rows:
    display(pd.DataFrame(url_rows).round(3))